# Hybrid Retrieval: Combining BM25 and Dense with RRF

BM25 and dense retrieval fail in *different* ways: BM25 misses paraphrases, dense retrieval can miss exact identifiers. **Hybrid retrieval** runs both and fuses their rankings with **Reciprocal Rank Fusion (RRF)**:

```
score(doc) = bm25_weight / (rank_bm25 + 60) + dense_weight / (rank_dense + 60)
```

RRF works on *ranks*, not raw scores, so it needs no score calibration between the two retrievers — a document ranked well by both wins; a document ranked well by only one still gets credit.

In [1]:
import sys
from pathlib import Path

import numpy as np

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from rag.data_ingestion import load_documents, chunk_documents
from rag.retrievers import BM25Retriever, DenseRetriever, HybridRetriever

PDF_PATH = ROOT / "data" / "google_10K.pdf"

documents = load_documents(PDF_PATH)
chunks = chunk_documents(documents, chunk_size=1000, chunk_overlap=100)

print(f"Loaded {len(documents)} pages -> {len(chunks)} chunks")

Loaded 107 pages -> 433 chunks


## RRF on a toy example

Two retrievers rank overlapping sets of five documents. Watch how RRF builds a consensus.

In [2]:
k = 60
bm25_ranking = ["doc_A", "doc_B", "doc_C", "doc_D", "doc_E"]
dense_ranking = ["doc_B", "doc_C", "doc_A", "doc_E", "doc_F"]

rrf_scores = {}
for doc_id in set(bm25_ranking) | set(dense_ranking):
    bm25_rank = bm25_ranking.index(doc_id) if doc_id in bm25_ranking else len(bm25_ranking)
    dense_rank = dense_ranking.index(doc_id) if doc_id in dense_ranking else len(dense_ranking)
    rrf_scores[doc_id] = (1.0 / (bm25_rank + k) + 1.0 / (dense_rank + k), bm25_rank, dense_rank)

print(f"BM25 ranking:  {bm25_ranking}")
print(f"Dense ranking: {dense_ranking}\n")

print(f"{'Rank':<6} {'Doc':<8} {'BM25 rank':<11} {'Dense rank':<12} {'RRF score':<10}")
print("-" * 50)
fused = sorted(rrf_scores.items(), key=lambda item: item[1][0], reverse=True)
for rank, (doc_id, (score, b, d)) in enumerate(fused, start=1):
    b_disp = str(b + 1) if b < len(bm25_ranking) else "-"
    d_disp = str(d + 1) if d < len(dense_ranking) else "-"
    print(f"{rank:<6} {doc_id:<8} {b_disp:<11} {d_disp:<12} {score:.6f}")

print("\ndoc_B tops the fused list: ranked highly by both retrievers.")

BM25 ranking:  ['doc_A', 'doc_B', 'doc_C', 'doc_D', 'doc_E']
Dense ranking: ['doc_B', 'doc_C', 'doc_A', 'doc_E', 'doc_F']

Rank   Doc      BM25 rank   Dense rank   RRF score 
--------------------------------------------------
1      doc_B    2           1            0.033060
2      doc_A    1           3            0.032796
3      doc_C    3           2            0.032522
4      doc_E    5           4            0.031498
5      doc_D    4           -            0.031258
6      doc_F    -           5            0.031010

doc_B tops the fused list: ranked highly by both retrievers.


## Building the hybrid retriever

`HybridRetriever` owns a `BM25Retriever` and a `DenseRetriever` internally. At query time each contributes its top `candidates_per_retriever` results, and RRF fuses them.

In [3]:
retriever = HybridRetriever(
    bm25_weight=1.0,
    dense_weight=1.0,
    candidates_per_retriever=40,
)
retriever.add_documents(chunks)

print(f"Indexed {len(retriever.bm25.documents)} chunks")
print(f"BM25 ready:  {retriever.bm25.bm25 is not None}")
print(f"Dense ready: {retriever.dense.embeddings is not None}")

Indexed 433 chunks
BM25 ready:  True
Dense ready: True


## Retrieval

In [4]:
query = "total revenues"

results = retriever.retrieve(query, top_k=5)

print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    preview = doc.page_content[:150].replace("\n", " ")
    print(f"{i}. [page {doc.metadata['page']}] {preview}...\n")

Query: total revenues

1. [page 66] vices 34,688  40,340  48,030  Google Services total 272,543  304,930  342,721  Google Cloud 33,088  43,229  58,705  Other Bets 1,527  1,648  1,537  He...

2. [page 39] he year ended December 31, 2025. • As of December 31, 2025, we had 190,820 employees. We are monitoring ongoing developments surrounding international...

3. [page 38] , 2024 2025 $ Change % Change Consolidated revenues $ 350,018  $ 402,836  $ 52,818  15 % Cost of revenues $ 146,306  $ 162,535  $ 16,229  11 % Operati...

4. [page 41] Table of Contents Alphabet Inc. Cost of Revenues The following table presents cost of revenues, including TAC (in millions, except percentages):   Yea...

5. [page 66] or less and cancellable contracts. Deferred Revenues We record deferred revenues when cash payments are received or due in advance of our performance,...



## BM25 vs Dense vs Hybrid

Retrieving the same query with all three strategies shows what each contributes. The hybrid list should draw from both source rankings.

In [5]:
bm25 = BM25Retriever()
bm25.add_documents(chunks)

dense = DenseRetriever()
dense.add_documents(chunks)

query = "How much revenue did the company generate?"

bm25_ids = [r.metadata["doc_id"] for r in bm25.retrieve(query, top_k=5)]
dense_ids = [r.metadata["doc_id"] for r in dense.retrieve(query, top_k=5)]
hybrid_ids = [r.metadata["doc_id"] for r in retriever.retrieve(query, top_k=5)]

print(f"Query: {query}\n")
print(f"BM25:   {bm25_ids}")
print(f"Dense:  {dense_ids}")
print(f"Hybrid: {hybrid_ids}\n")

print(f"BM25 ∩ Dense:    {set(bm25_ids) & set(dense_ids) or '{}'}")
print(f"Hybrid ∩ BM25:   {set(hybrid_ids) & set(bm25_ids) or '{}'}")
print(f"Hybrid ∩ Dense:  {set(hybrid_ids) & set(dense_ids) or '{}'}")

Query: How much revenue did the company generate?

BM25:   ['chunk_47', 'chunk_236', 'chunk_295', 'chunk_24', 'chunk_4']
Dense:  ['chunk_190', 'chunk_179', 'chunk_189', 'chunk_246', 'chunk_184']
Hybrid: ['chunk_289', 'chunk_295', 'chunk_47', 'chunk_190', 'chunk_236']

BM25 ∩ Dense:    {}
Hybrid ∩ BM25:   {'chunk_236', 'chunk_47', 'chunk_295'}
Hybrid ∩ Dense:  {'chunk_190'}


## Tuning the weights

`bm25_weight` and `dense_weight` shift the balance between exact matching and semantics. The weights only affect the fusion step, so we can adjust them on the already-indexed retriever without re-embedding anything.

In [6]:
configs = [
    (1.0, 1.0, "Equal (default)"),
    (2.0, 1.0, "Favor BM25 — exact-match queries"),
    (1.0, 2.0, "Favor Dense — semantic queries"),
]

query = "annual financial results"

print(f"Query: {query}\n")
print(f"{'Config':<36} Top-3 doc IDs")
print("-" * 75)
for bm25_w, dense_w, label in configs:
    retriever.bm25_weight = bm25_w
    retriever.dense_weight = dense_w
    ids = [r.metadata["doc_id"] for r in retriever.retrieve(query, top_k=3)]
    print(f"{label:<36} {ids}")

retriever.bm25_weight = 1.0
retriever.dense_weight = 1.0

Query: annual financial results

Config                               Top-3 doc IDs
---------------------------------------------------------------------------
Equal (default)                      ['chunk_157', 'chunk_183', 'chunk_232']
Favor BM25 — exact-match queries     ['chunk_157', 'chunk_183', 'chunk_232']
Favor Dense — semantic queries       ['chunk_157', 'chunk_232', 'chunk_183']


## Takeaways

- RRF fuses rankings without needing comparable scores between retrievers.
- Hybrid retrieval is a robust default: it rarely loses badly to either component.
- Weights let you bias toward keywords or semantics per workload.

`04_reranker.ipynb` adds a second stage that rescores the candidates with a cross-encoder.